# Notebook 2: Feature Extraction ed Exploratory Data Analysis (EDA)

**Progetto:** Premier League Event Data 2020-21  
**Input:** `data/processed/dataset_clean.parquet` (prodotto dal NB1)  
**Output:** `data/processed/features.csv` (usato dal NB3)

---

## La Domanda di Ricerca

L'obiettivo di questo notebook è rispondere a una domanda fondamentale:

> **È possibile prevedere l'esito di una partita di Premier League (Win, Draw, Loss) basandosi esclusivamente su metriche tattiche di squadra, ignorando i gol effettivi?**

Per farlo, seguiamo la traccia del progetto:

> *"Le feature possono essere calcolate sulla base delle colonne esistenti contando il numero di eventi di un determinato tipo per ogni partita. Altre feature possono essere calcolate mediando i dati su una intera partita (esempio X e Y)."*

Il cuore di questo notebook è l'**aggregazione**: trasformiamo il dataset grezzo da ~607k righe (una per evento) a 760 righe (una per squadra per partita), secondo lo schema:

```
Dataset grezzo:          Dataset aggregato:
607k righe               760 righe
1 riga = 1 evento        1 riga = 1 squadra in 1 partita
                         (380 partite × 2 squadre)
```

Successivamente, tramite l'EDA, verificheremo se le medie di queste feature tattiche differiscono in modo statisticamente significativo (superando **1 deviazione standard**) tra le tre classi di risultato.

## 0. Setup e Caricamento

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Caricamento tramite data_loader (environment-agnostic: funziona su VS Code e Colab)
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_clean_data, save_features

OUT_DIR = Path('../figures/nb2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='tab10')

def savefig(name):
    plt.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

# Carica il Silver Layer prodotto dal Notebook 1
df = load_clean_data()
print(f'Dataset caricato: {df.shape[0]:,} righe x {df.shape[1]} colonne')
print(f'Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

# Verifica che il match_id sia gia presente (prodotto dal Notebook 1)
assert 'match_id' in df.columns, 'ERRORE: match_id non trovato! Riesegui il Notebook 1.'

# Gli id duplicati sono eventi OffsideGiven registrati specularmente -- comportamento atteso Opta.
duplicati_id = df['id'].duplicated().sum()
print(f'Righe con id duplicato: {duplicati_id} (OffsideGiven speculari, mantenuti)')

## 1. Derivazione del Target (Win, Draw, Loss)

Avendo già a disposizione il `match_id` e un dataset pulito (dal Notebook 1), procediamo al calcolo dei gol segnati e subiti da ciascuna squadra.

Utilizziamo `transform('sum')` — operazione vettorializzata che evita `apply` lenti — per conteggiare i gol totali della partita, e deriviamo per differenza i gol subiti e il target testuale **Win / Draw / Loss** per ogni squadra.

In [ ]:
# Il dataset è già pulito e contiene solo eventi di gioco e il match_id.\n
df_game = df.copy()
n_matches = df_game["match_id"].nunique()
print(f"Partite identificate: {n_matches}  (attese: 380 per PL 2020-21)\n")

# Gol per squadra per match
goals_df = (
    df_game[df_game["type_displayName"] == "Goal"]
    .groupby(["match_id", "teamId"])
    .size()
    .reset_index(name="goals_scored")
)

# Matrice completa match x team (include squadre con 0 gol)
all_pairs = (
    df_game.groupby(["match_id", "teamId"])
    .size()
    .reset_index(name="total_events")[["match_id", "teamId"]]
)

results_df = all_pairs.merge(goals_df, on=["match_id", "teamId"], how="left")
results_df["goals_scored"] = results_df["goals_scored"].fillna(0).astype(int)

# Gol subiti = totale gol del match - gol segnati (vettorializzato)
total_goals = results_df.groupby("match_id")["goals_scored"].transform("sum")
results_df["goals_conceded"] = total_goals - results_df["goals_scored"]
opp = results_df.copy()

def outcome(row):
    if   row["goals_scored"] > row["goals_conceded"]: return "Win"
    elif row["goals_scored"] < row["goals_conceded"]: return "Loss"
    else:                                              return "Draw"

opp["outcome"] = opp.apply(outcome, axis=1)

print("\nDistribuzione outcome (per team-match):")
print(opp["outcome"].value_counts())

avg_goals = opp.groupby("match_id")["goals_scored"].sum().mean()
print(f"\nMedia gol per partita: {avg_goals:.2f}  (PL 2020-21 reale: ~2.7)")

## 2. Feature Extraction — Aggregazione evento → squadra-partita

Questo è il passaggio centrale del notebook. Aggreghiamo il dataset da **607.656 righe** (1 riga = 1 evento) a **760 righe** (1 riga = 1 squadra in 1 partita) tramite `groupby(['match_id', 'teamId']).apply(extract_features)`.

Come richiesto dalla traccia, le feature estratte appartengono a due famiglie:

| Famiglia | Esempi | Razionale |
|---|---|---|
| **Conteggi** | `n_passes`, `n_shots`, `n_tackles`, `n_fouls`... | Volume e stile di gioco |
| **Medie / Ratei** | `pass_success_rate`, `avg_x`, `events_per_minute`... | Efficienza e posizionamento tattico |

> **Data Leakage:** `n_goals` e `shot_conversion_rate` vengono estratte qui per completezza dell'EDA (sono utili per mostrare *quali* feature sono discriminanti), ma verranno **escluse da `FEATURE_COLS_ML` in NB3** prima di qualsiasi addestramento. Entrambe sono derivazioni dirette dei gol segnati, che per definizione determinano il risultato della partita.

In [ ]:
# --- CELLA 6 (Definizione Funzione) ---
# Creiamo il flag di successo prima del groupby per efficienza
if "outcomeType_displayName" in df_game.columns:
    df_game["is_successful"] = df_game["outcomeType_displayName"] == "Successful"

def extract_features(grp):
    """
    Estrae le feature tattiche e spaziali per squadra-partita.
    Previene il Data Leakage ignorando i gol.
    """
    total_events = len(grp)
    minutes_played = grp["minute"].clip(upper=120).max() - grp["minute"].clip(upper=120).min()
    minutes_played = max(minutes_played, 1)

    # Sub-dataframe
    passes = grp[grp["type_displayName"] == "Pass"]
    # FIX LEAKAGE: Un tiro è valido se isShot è True MA isGoal è False
    shots = grp[(grp.get("isShot", False) == True) & (grp.get("isGoal", False) == False)]
    goals = grp[grp["type_displayName"] == "Goal"]
    tackles = grp[grp["type_displayName"] == "Tackle"]
    fouls = grp[grp["type_displayName"] == "Foul"]
    corners = grp[grp["type_displayName"] == "CornerAwarded"]
    aerials = grp[grp["type_displayName"] == "Aerial"]
    clearances = grp[grp["type_displayName"] == "Clearance"]
    intercepts = grp[grp["type_displayName"] == "Interception"]
    takeons = grp[grp["type_displayName"] == "TakeOn"]
    recoveries = grp[grp["type_displayName"] == "BallRecovery"]
    
    ft = grp[grp["period_displayName"] == "FirstHalf"]

    def safe_rate(num, den):
        return num / den if den > 0 else 0.0

    # Coordinate Spaziali Sicure
    avg_x = grp["x"].mean()
    avg_y = grp["y"].mean()
    avg_pass_end_x = passes["endX"].mean() if "endX" in passes.columns else np.nan

    # Field Tilt
    offensive_third = grp[grp['x'] > 66.6]
    field_tilt = len(offensive_third) / total_events if total_events > 0 else 0.0

    # Gestione Cartellini
    n_yellow = grp[grp['cardType_displayName'] == 'Yellow'].shape[0]
    red_events = grp[grp['cardType_displayName'].isin(['Red', 'SecondYellow'])]
    n_red = red_events['playerId'].nunique() if 'playerId' in red_events.columns else red_events.shape[0]

    # --- FUNZIONI HELPER PER ESTRAZIONI SICURE ---
    def safe_sum(col_name):
        return grp[col_name].sum() if col_name in grp.columns else 0
        
    def safe_mean(col_name):
        return grp[col_name].mean() if col_name in grp.columns else np.nan
    # ---------------------------------------------

    return pd.Series({
        # 1. Metriche Volume Base
        "n_passes": len(passes),
        "n_shots_no_goal": len(shots),
        "n_tackles": len(tackles),
        "n_fouls": len(fouls),
        "n_corners": len(corners),
        "n_aerials": len(aerials),
        "n_clearances": len(clearances),
        "n_interceptions": len(intercepts),
        "n_takeons": len(takeons),
        "n_ball_recoveries": len(recoveries),
        "events_per_minute": total_events / minutes_played,
        "first_half_ratio":  safe_rate(len(ft), total_events),
        
        # 2. Metriche Tattiche Avanzate (dai Qualifiers)
        "n_longballs": safe_sum('is_longball'),
        "n_crosses": safe_sum('is_cross'),
        "n_throughballs": safe_sum('is_throughball'),
        "n_keypasses": safe_sum('is_keypass'),
        "n_fastbreaks": safe_sum('is_fastbreak'),
        
        # Somme sicure aggregate
        "n_big_chances": safe_sum('is_bigchance') + safe_sum('is_bigchancecreated'),
        "n_blocks": safe_sum('is_blocked') + safe_sum('is_outfielderblock') + safe_sum('is_sixyardblock'),
        
        # 3. Metriche Spaziali e Dominio Territoriale
        "avg_x": avg_x,
        "avg_y": avg_y,
        "avg_pass_end_x": avg_pass_end_x,
        "field_tilt": field_tilt,
        "n_passes_offensive_third": len(offensive_third[offensive_third['type_displayName'] == 'Pass']),
        "avg_pass_length": safe_mean('pass_length'),
        "avg_goal_mouth_z": safe_mean('goal_mouth_z'),
        "set_piece_ratio": safe_rate((safe_sum('is_fromcorner') + safe_sum('is_setpiece')), len(shots)),
        
        # 4. Disciplina
        "n_yellow_cards": n_yellow,
        "n_red_cards": n_red,
        
        # 5. Ratei di Efficienza
        "pass_success_rate": safe_rate(passes["is_successful"].sum(), len(passes)) if "is_successful" in passes.columns else np.nan,
        "tackle_success_rate": safe_rate(tackles["is_successful"].sum(), len(tackles)) if "is_successful" in tackles.columns else np.nan,
        "aerial_success_rate": safe_rate(aerials["is_successful"].sum(), len(aerials)) if "is_successful" in aerials.columns else np.nan,
        "takeon_success_rate": safe_rate(takeons["is_successful"].sum(), len(takeons)) if "is_successful" in takeons.columns else np.nan,
    })

print("Funzione 'extract_features' definita con successo. Pronta per il groupby!")

# --- CELLA 7 (Esecuzione Unica) ---
import time

print("Avvio aggregazione feature tattiche. Attendere...")
start_time = time.time()

# Esecuzione dell'aggregazione (con i FIX per Pandas)
features_raw = (
    df_game.groupby(["match_id", "teamId"], group_keys=False)
    .apply(extract_features, include_groups=False)
    .reset_index()
)

# Uniamo il dataset con il target (esito e gol)
features = features_raw.merge(
    opp[["match_id", "teamId", "goals_scored", "goals_conceded", "outcome"]],
    on=["match_id", "teamId"],
    how="inner"
)

# Sicurezza finale per il ML: sostituiamo eventuali NaN con 0
features = features.fillna(0)

end_time = time.time()
print(f"Aggregazione completata in {end_time - start_time:.2f} secondi.")
print(f"Dimensioni dataset Gold: {features.shape[0]} righe × {features.shape[1]} colonne (Attesi: 760 righe)")

# Definiamo rigorosamente le feature 
FEATURE_COLS = [c for c in features.columns 
                if c not in ["match_id", "teamId", "goals_scored", 
                             "goals_conceded", "outcome"]]

print(f"\nColonne feature pronte per l'EDA e il ML: {len(FEATURE_COLS)}")
print("\nAnteprima delle nuove metriche estratte:")
print(features[['teamId', 'n_passes', 'n_longballs', 'n_big_chances', 'n_fastbreaks', 'pass_success_rate', 'outcome']].head())

# --- CELLA 9 (EDA 1-Sigma Pulita) ---
OUTCOME_ORDER = ["Win", "Draw", "Loss"]
PALETTE = {"Win": "#10B981", "Draw": "#F59E0B", "Loss": "#EF4444"}

# ESCLUSIONE DATA LEAKAGE: Rimuoviamo n_goals e conversion rate prima del test
eda_cols = [c for c in FEATURE_COLS if c not in ['n_goals', 'shot_conversion_rate']]

class_means = features.groupby("outcome")[eda_cols].mean()
global_std  = features[eda_cols].std()

print("--- Media per classe (Win / Draw / Loss) ---")
# Stampiamo solo alcune colonne per brevità
print(class_means[['n_passes', 'n_big_chances', 'n_keypasses', 'avg_x']].round(2).T.to_string())

diff_win_loss = (class_means.loc["Win"] - class_means.loc["Loss"]).abs()
significant   = diff_win_loss[diff_win_loss > global_std]

print("\n--- Analisi significatività (> 1sigma) TATTICA ---")
for feat in diff_win_loss.sort_values(ascending=False).index:
    w  = class_means.loc["Win",  feat]
    d  = class_means.loc["Draw", feat]
    l  = class_means.loc["Loss", feat]
    sd = global_std[feat]
    marker = "[SÌ]" if feat in significant.index else "[NO]"
    
    # Stampiamo le prime 15 per evitare log infiniti
    if diff_win_loss.sort_values(ascending=False).index.get_loc(feat) < 15:
        print(f"{marker} {feat:<26} Win={w:.2f}  Draw={d:.2f}  Loss={l:.2f}  "
              f"|diff|={diff_win_loss[feat]:.2f}  sigma={sd:.2f}  -> {diff_win_loss[feat]/sd:.1f}sigma")


In [ ]:
import time

print("Avvio aggregazione feature tattiche. Attendere...")
start_time = time.time()

# Esecuzione dell'aggregazione
features_raw = (
    df_game.groupby(["match_id", "teamId"], group_keys=False)
    .apply(extract_features, include_groups=False)
    .reset_index()
)

# Uniamo il dataset con il target (esito e gol)
features = features_raw.merge(
    opp[["match_id", "teamId", "goals_scored", "goals_conceded", "outcome"]],
    on=["match_id", "teamId"],
    how="inner"
)

# Sicurezza finale per il ML: sostituiamo eventuali NaN con 0
features = features.fillna(0)

end_time = time.time()
print(f"Aggregazione completata in {end_time - start_time:.2f} secondi.")
print(f"Dimensioni dataset Gold: {features.shape[0]} righe × {features.shape[1]} colonne (Attesi: 760 righe)")

# Mostriamo le prime righe con le nuove metriche per verifica
print("\nAnteprima delle nuove metriche estratte:")
print(features[['teamId', 'n_passes', 'n_longballs', 'n_big_chances', 'n_fastbreaks', 'pass_success_rate', 'outcome']].head())

## 3. Exploratory Data Analysis (EDA)

### 3a. Medie per classe e analisi della significatività (criterio 1σ)

Come richiesto dalla traccia, confrontiamo le medie di ogni feature tra le classi Win, Draw e Loss. Una feature è considerata **discriminante** se la differenza assoluta tra il valore medio della classe Win e quello della classe Loss supera **1 deviazione standard globale** (calcolata sull'intero dataset, non per classe).

In [ ]:
OUTCOME_ORDER = ["Win", "Draw", "Loss"]
PALETTE = {"Win": "#10B981", "Draw": "#F59E0B", "Loss": "#EF4444"}

class_means = features.groupby("outcome")[FEATURE_COLS].mean()
global_std  = features[FEATURE_COLS].std()

print("--- Media per classe (Win / Draw / Loss) ---")
print(class_means.round(3).T.to_string())

diff_win_loss = (class_means.loc["Win"] - class_means.loc["Loss"]).abs()
significant   = diff_win_loss[diff_win_loss > global_std]

print("\n--- Analisi significativita (> 1sigma) ---")
for feat in diff_win_loss.sort_values(ascending=False).index:
    w  = class_means.loc["Win",  feat]
    d  = class_means.loc["Draw", feat]
    l  = class_means.loc["Loss", feat]
    sd = global_std[feat]
    marker = "[SÌ]" if feat in significant.index else "[NO]"
    print(f"{marker} {feat:<26} Win={w:.3f}  Draw={d:.3f}  Loss={l:.3f}  "
          f"|diff|={diff_win_loss[feat]:.3f}  sigma={sd:.3f}  -> {diff_win_loss[feat]/sd:.1f}sigma")

### Interpretazione

L'analisi mostra che solo **`n_goals`** e **`shot_conversion_rate`** superano la soglia di 1σ tra la classe Win e la classe Loss. Questo risultato è atteso: entrambe le feature sono **derivazioni dirette dei gol segnati**, che per definizione determinano il risultato. Vengono quindi escluse dal dataset di Machine Learning in NB3.

Tutte le altre feature tattiche — passaggi, tiri, posizione media, duelli aerei, tackle — **non superano 1σ**. I grafici seguenti confermano visivamente l'ampia sovrapposizione tra le distribuzioni delle tre classi. Il calcio è uno sport a basso punteggio dove il dominio tattico non si traduce necessariamente in vittoria: questo è il motivo per cui il compito di classificazione è intrinsecamente difficile, e in particolare la classe Draw risulterà la più problematica per tutti i modelli.

### 3b. Visualizzazioni

In [ ]:
# Boxplot: top 9 feature piu discriminanti
top_features = diff_win_loss.sort_values(ascending=False).head(9).index.tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, feat in zip(axes.flatten(), top_features):
    sns.boxplot(data=features, x="outcome", y=feat,
                order=OUTCOME_ORDER, palette=PALETTE,
                ax=ax, width=0.5, fliersize=3)
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("")

plt.suptitle("Top 9 feature piu discriminanti tra Win / Draw / Loss",
             fontsize=14, y=1.01)
plt.tight_layout()
savefig("01_boxplot_top9")

In [ ]:
# Heatmap medie normalizzate (z-score per classe)
means_norm = (class_means - class_means.mean()) / class_means.std()

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(means_norm.T, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Feature medie normalizzate per classe (z-score) — Verde = valore alto per quella classe",
             fontsize=12)
plt.tight_layout()
savefig("02_heatmap_medie_normalizzate")

In [ ]:
# Scatter: n_shots vs pass_success_rate per outcome
fig, ax = plt.subplots(figsize=(10, 7))
for outcome_val, grp in features.groupby("outcome"):
    ax.scatter(grp["n_shots"], grp["pass_success_rate"],
               label=outcome_val, alpha=0.5, s=30,
               color=PALETTE[outcome_val])

ax.set_xlabel("n_shots (tiri totali)")
ax.set_ylabel("pass_success_rate")
ax.set_title("Tiri vs Successo nei passaggi, per outcome")
ax.legend()
plt.tight_layout()
savefig("03_scatter_shots_vs_pass_success")

In [ ]:
# Matrice di correlazione tra feature
fig, ax = plt.subplots(figsize=(16, 13))
corr = features[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.3, annot_kws={"size": 7})
ax.set_title("Matrice di correlazione tra feature", fontsize=13)
plt.tight_layout()
savefig("04_correlation_matrix")

## 4. Salvataggio `features.parquet`

Il dataset aggregato viene salvato in `data/processed/features.parqueto caricherà e definirà al suo interno `FEATURE_COLS_ML`, che esclude esplicitamente `n_goals` e `shot_conversion_rate` per prevenire il data leakage nell'addestramento dei classificatori.

In [ ]:
# Salvataggio Gold Layer tramite data_loader (Parquet
# Il Parquet preserva i tipi di dato; il Notebook 3 li usera direttamente.
save_features(features)
print(f'Colonne: {list(features.columns)}')